In [ ]:
# v6: 2D CNN on spectrograms — EfficientNet-B0 + KL loss + early stopping
!pip install timm -q

In [ ]:
import sys, os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score

# ── src imports from uploaded code dataset ──────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    CODE_DIR = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code'
else:
    CODE_DIR = os.path.abspath('../kaggle_upload')

sys.path.insert(0, CODE_DIR)
sys.path.insert(0, os.path.join(CODE_DIR, 'src'))

from src.config_2d  import Config2D
from src.dataset_2d import SpectrogramDataset
from src.model_2d   import EfficientNetEEG

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | torch {torch.__version__}")

In [ ]:
cfg = Config2D()

if IS_KAGGLE:
    DATA_ROOT = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    # train_test_split.csv (not train.csv) — contains inner_fold and split columns
    cfg.metadata_csv    = '/kaggle/input/datasets/xiaosufrankhu/hms-eeg-code/train_test_split.csv'
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')
else:
    DATA_ROOT = os.path.abspath('../')
    cfg.metadata_csv    = os.path.abspath('../data_meta_splits/train_test_split.csv')
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')

# ── reproducibility ─────────────────────────────────────────────────────────
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE  = torch.device(cfg.device)
USE_AMP = DEVICE.type == 'cuda'

print(cfg)
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

In [ ]:
train_ds = SpectrogramDataset(cfg.metadata_csv, cfg.spectrogram_dir, cfg, split='train')
val_ds   = SpectrogramDataset(cfg.metadata_csv, cfg.spectrogram_dir, cfg, split='val')

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=True,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=True,
)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")

# batch shape sanity check
batch = next(iter(train_loader))
print(f"image      : {tuple(batch['image'].shape)}")
print(f"soft_label : {tuple(batch['soft_label'].shape)}")
print(f"label      : {tuple(batch['label'].shape)}")

In [ ]:
model = EfficientNetEEG(
    backbone    = cfg.backbone,
    num_classes = cfg.num_classes,
    pretrained  = cfg.pretrained,
).to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone    : {cfg.backbone}")
print(f"Parameters  — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
# v6: 2D spectrogram CNN + KL loss + cosine LR + early stopping
GRAD_CLIP = 1.0

criterion     = nn.KLDivLoss(reduction='batchmean')
val_criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.num_epochs)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

best_f1, best_epoch, history = 0.0, 0, []
wait = 0

for epoch in range(1, cfg.num_epochs + 1):
    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    t0         = time.time()
    train_loss = 0.0

    for batch in train_loader:
        x    = batch['image'].to(DEVICE)       # (B, 4, H, W)
        soft = batch['soft_label'].to(DEVICE)  # (B, 6)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits    = model(x)
            log_probs = F.log_softmax(logits, dim=1)
            loss      = criterion(log_probs, soft)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()

    # ── validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_ce, val_kl = 0.0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            x    = batch['image'].to(DEVICE)
            y    = batch['label'].to(DEVICE)       # hard label
            soft = batch['soft_label'].to(DEVICE)

            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(x)

            val_ce += val_criterion(logits, y).item()
            val_kl += criterion(F.log_softmax(logits, dim=1), soft).item()
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    avg_train = train_loss / len(train_loader)
    avg_val   = val_ce     / len(val_loader)
    avg_kl    = val_kl     / len(val_loader)
    elapsed   = time.time() - t0

    history.append({
        'epoch': epoch, 'train_kl': avg_train,
        'val_kl': avg_kl, 'val_ce': avg_val, 'macro_f1': macro_f1,
    })

    print(f"Epoch {epoch:03d} | train_kl {avg_train:.4f} | "
          f"val_kl {avg_kl:.4f} | val_ce {avg_val:.4f} | "
          f"macro_f1 {macro_f1:.4f} | {elapsed:.0f}s")

    # ── early stopping (monitor macro_f1, identical to v3) ───────────────────
    if macro_f1 > best_f1:
        best_f1, best_epoch = macro_f1, epoch
        wait = 0
        torch.save(model.state_dict(), 'best_model_v6.pt')
        print(f"  ✓ saved best model (f1={best_f1:.4f})")
    else:
        wait += 1
        if wait >= cfg.patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {cfg.patience} epochs)")
            break

print(f"\nTraining complete. Best macro_f1: {best_f1:.4f} (epoch {best_epoch})")

In [ ]:
epochs   = [h['epoch']    for h in history]
train_kl = [h['train_kl'] for h in history]
val_kl   = [h['val_kl']   for h in history]
val_ce   = [h['val_ce']   for h in history]
macro_f1 = [h['macro_f1'] for h in history]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(epochs, train_kl, label='train KL')
ax1.plot(epochs, val_kl,   label='val KL')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('KL Divergence')
ax1.set_title('KL Divergence (train vs val)'); ax1.legend()

ax2.plot(epochs, val_ce, color='orange', label='val CE')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Cross-Entropy Loss')
ax2.set_title('Val CE Loss (monitoring)'); ax2.legend()

ax3.plot(epochs, macro_f1, color='green')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Macro F1')
ax3.set_title('Validation Macro F1')

plt.tight_layout()
plt.savefig('training_curves_v6.png', dpi=150)
plt.show()

print(f"\nFinal epoch summary:")
print(f"  Best macro_f1 : {max(macro_f1):.4f}  (epoch {epochs[macro_f1.index(max(macro_f1))]})") 
print(f"  Best val KL   : {min(val_kl):.4f}  (epoch {epochs[val_kl.index(min(val_kl))]})") 
print(f"  Final train KL: {train_kl[-1]:.4f}")
print(f"  Final val KL  : {val_kl[-1]:.4f}")